In [0]:
import json
from pyspark.sql.types import *
from pyspark.sql.types import StructType, StructField, StringType

In [0]:
jira_schema = StructType([
    StructField("expand", StringType()),
    StructField("id", StringType()),
    StructField("key", StringType()),
   
    StructField("fields",StructType([

         StructField("summary", StringType()),

        StructField("description", StructType([
            StructField("type", StringType()),
            StructField("version", StringType())
        ])),

        StructField("issuetype", StructType([
            StructField("id", StringType()),
            StructField("name", StringType()),
            StructField("description", StringType()),
            StructField("iconUrl", StringType()),
            StructField("self", StringType())
        ])),

        StructField("status", StructType([
            StructField("id", StringType()),
            StructField("name", StringType()),
            StructField("description", StringType()),
            StructField("iconUrl", StringType()),
            StructField("self", StringType()),

            StructField("statusCategory", StructType([
                StructField("id", StringType()),
                StructField("key", StringType()),
                StructField("name", StringType()),
                StructField("colorName", StringType()),
                StructField("self", StringType())
            ]))
        ])),

        StructField("assignee", StructType([
            StructField("accountId", StringType()),
            StructField("displayName", StringType()),
            StructField("emailAddress", StringType()),
            StructField("accountType", StringType()),
            StructField("timeZone", StringType()),
            StructField("self", StringType())
        ])),

        StructField("reporter", StructType([
            StructField("accountId", StringType()),
            StructField("displayName", StringType()),
            StructField("emailAddress", StringType()),
            StructField("accountType", StringType()),
            StructField("timeZone", StringType()),
            StructField("self", StringType())
        ])),

        StructField("priority", StructType([
            StructField("id", StringType()),
            StructField("name", StringType()),
            StructField("iconUrl", StringType()),
            StructField("self", StringType())
        ])),

        StructField("parent", StructType([
            StructField("id", StringType()),
            StructField("key", StringType()),
            StructField("self", StringType())
        ])),

        StructField("customfield_10020", ArrayType(
            StructType([
                StructField("id", StringType()),
                StructField("name", StringType()),
                StructField("state", StringType()),
                StructField("startDate", StringType()),
                StructField("endDate", StringType())
            ])
        )),

        StructField("customfield_10016", DoubleType())

    ]))
])



In [0]:
def parse_issue(issue):   # To be used in issues_bronze for parsing
    return{"raw_json": json.dumps(issue)}

In [0]:

def parse_project(project):
    return project


In [0]:

project_schema = StructType([
    StructField("id", StringType()),
    StructField("key", StringType()),
    StructField("name", StringType()),
    StructField("projectTypeKey", StringType()),
    StructField("self", StringType())
])


In [0]:

PROJECTS_CONFIG = {
    "name"               : "projects",
    "version_key": "version", 
    "endpoint_key"       : "projects",
    "data_key"           : "values",
    "strategy"           : "flat",
    "fields"             : "",
    "schema"             : None,
    "parser"             : lambda x: {"raw_json": json.dumps(x)},
    "df_schema"          : StructType([        
        StructField("raw_json", StringType(), True)
    ]),
    "result_fields"      : ["key"],
    "raw_table"          : None,
    "selected_table"     : None,
    "selected_columns"   : [],
    "raw_write_mode"     : "append",
    "selected_write_mode": "append",
    "merge_key"          : None,
}

ISSUES_CONFIG = {
    "name": "issues",
    "endpoint_key": "issues",
    "version_key": "version", 
    "data_key": "issues",
    "strategy": "per_parent",
    "raw_write_mode"     : "append",
    "selected_write_mode": "merge",
    "merge_key"          : "id",

    "parent": {
        "source": "projects",            
        "field": "key",                  
        "param_name": "jql",             
        "param_template": 'project = "{VALUE}"'
    },

    "fields": "summary,description,issuetype,status,assignee,reporter,priority,parent,customfield_10020,customfield_10016",

    "schema": jira_schema,
    "parser": parse_issue,

    "raw_table": "issues_raw",
    "selected_table": "issues_selected",

    "selected_columns": [
        ("parsed.id", "id"),
        ("parsed.key", "key"),
        ("parsed.fields.summary", "summary"),
        ("parsed.fields.status.name", "status"),
        ("parsed.fields.assignee.displayName", "assignee")
    ]
}